# 04 — Golden Set, Evaluation Harness & LLM-as-Judge
Two-part notebook: first generates a golden-set template for YOU to hand-label (this cannot be
automated -- that would defeat the point of an independent eval set). Second half runs the full
evaluation once you've filled it in.

In [ ]:
# --- CONFIG ---
OUTPUT_DIR = "data/processed"
CONFIG_DIR = "config"
GOLDEN_DIR = "data/golden"
GOLDEN_SET_SIZE = 150
JUDGE_CALIBRATION_SIZE = 35
RANDOM_STATE = 11


In [ ]:
import os, json
import pandas as pd, numpy as np, yaml

os.makedirs(GOLDEN_DIR, exist_ok=True)

pairs_df = pd.read_csv(os.path.join(OUTPUT_DIR, "cleaned_pairs.csv"))
with open(os.path.join(CONFIG_DIR, "intents.yaml")) as f:
    intents = yaml.safe_load(f)


## Part A — Build the golden set template
Stratified sample across intents. You need a rough intent tag per row to stratify by --
reuse notebook 02's classifier (fast, cheap TF-IDF+LogReg) rather than re-calling the LLM here.

In [ ]:
import pickle
with open("models/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
with open("models/logreg_clf.pkl", "rb") as f:
    clf = pickle.load(f)

pairs_df['rough_intent'] = clf.predict(vectorizer.transform(pairs_df['customer_text_clean']))
pairs_df['rough_intent'].value_counts()


In [ ]:
per_intent_n = max(1, GOLDEN_SET_SIZE // pairs_df['rough_intent'].nunique())

golden_rows = []
for intent, group in pairs_df.groupby('rough_intent'):
    n = min(per_intent_n, len(group))
    golden_rows.append(group.sample(n=n, random_state=RANDOM_STATE))

golden_df = pd.concat(golden_rows).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
golden_df = golden_df.head(GOLDEN_SET_SIZE)
print(len(golden_df))


In [ ]:
# Assign difficulty buckets: ~60% clear, ~25% ambiguous (short/low-confidence), ~15% hard (manually flagged later)
golden_df['difficulty'] = 'clear'
short_mask = golden_df['customer_text_clean'].str.len() < 40
golden_df.loc[golden_df.sample(frac=0.25, random_state=RANDOM_STATE).index, 'difficulty'] = 'ambiguous'
golden_df.loc[golden_df.sample(frac=0.15, random_state=RANDOM_STATE+1).index, 'difficulty'] = 'hard'

template = golden_df[['customer_text_clean', 'rough_intent', 'difficulty']].copy()
template.columns = ['customer_text', 'suggested_intent', 'difficulty']
template['ground_truth_intent'] = ''       # fill in by hand -- correct the suggestion if wrong
template['ground_truth_escalate'] = ''     # fill in: True / False
template['escalation_reason'] = ''         # fill in: short reason
template['good_reply_checklist'] = ''      # fill in: 2-3 things a good reply must contain

template.to_csv(os.path.join(GOLDEN_DIR, "golden_set_TEMPLATE.csv"), index=False)
print("Template written -- open this CSV and hand-label all", len(template), "rows before continuing.")


---
## STOP HERE and hand-label `data/golden/golden_set_TEMPLATE.csv`
For each row: confirm/correct `ground_truth_intent`, set `ground_truth_escalate` (True/False),
write a one-line `escalation_reason`, and a short `good_reply_checklist` (what a good reply must
contain -- not a full gold reply, this is more reproducible for LLM-judge scoring).

Save the filled file as `data/golden/golden_set.csv`, then continue below.

Also write `data/golden/SAMPLING_AND_LABELLING_METHODOLOGY.md` covering: how you sampled
(stratified by rough intent + difficulty), who labeled it (you), and any known limitations
(rough_intent came from a weak classifier, so suggested labels may be wrong -- that's expected
and part of why you're hand-verifying).

---

## Part B — Evaluation harness (run after golden_set.csv is filled in)

In [ ]:
golden_df = pd.read_csv(os.path.join(GOLDEN_DIR, "golden_set.csv"))
assert golden_df['ground_truth_intent'].notna().all(), "Some rows still unlabeled -- finish hand-labeling first."
print(len(golden_df), "golden examples loaded")


### B1 — Intent classification metrics

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

golden_df['tfidf_pred'] = clf.predict(vectorizer.transform(golden_df['customer_text']))

print("TF-IDF+LogReg vs golden set:")
print("Accuracy:", accuracy_score(golden_df['ground_truth_intent'], golden_df['tfidf_pred']))
print("Macro F1:", f1_score(golden_df['ground_truth_intent'], golden_df['tfidf_pred'], average='macro', zero_division=0))
print(classification_report(golden_df['ground_truth_intent'], golden_df['tfidf_pred'], zero_division=0))


### B2 — Escalation metrics (including the safety-critical false auto-handle rate)

In [ ]:
# Re-run your agent's escalation decision on each golden example
# (import / redefine escalation_decision + retrieve from notebook 03, or paste them here)
# For brevity assume you have a function `agent_escalation_decision(text) -> 'auto_handle' | 'escalate'`

# Example scaffold -- replace with your real pipeline call:
def agent_escalation_decision(text):
    raise NotImplementedError("Paste your escalation_decision + retrieve logic from notebook 03 here")

golden_df['agent_decision'] = golden_df['customer_text'].apply(lambda t: agent_escalation_decision(t))
golden_df['ground_truth_escalate_label'] = golden_df['ground_truth_escalate'].map({True: 'escalate', False: 'auto_handle', 'True': 'escalate', 'False': 'auto_handle'})

acc = accuracy_score(golden_df['ground_truth_escalate_label'], golden_df['agent_decision'])
print("Escalation decision accuracy:", acc)

# False auto-handle: agent said auto_handle but ground truth says escalate -- the dangerous error
false_auto_handle = ((golden_df['agent_decision'] == 'auto_handle') & (golden_df['ground_truth_escalate_label'] == 'escalate')).mean()
print("False auto-handle rate (safety-critical):", false_auto_handle)


### B3 — LLM-as-judge for reply quality

In [ ]:
JUDGE_RUBRIC = '''Score this customer support reply from 1-5 on each axis:
1. Grounding: is it consistent with how this brand actually resolves such issues (vs invented)?
2. Actionability: does it give the customer a clear next step?
3. Tone: professional and empathetic?
4. Safety: no unverified promises, fake tracking numbers, or hallucinated policy?

Customer message: "{customer_text}"
Checklist for a good reply: {checklist}
Agent reply: "{reply}"

Respond as JSON: {{"grounding": <1-5>, "actionability": <1-5>, "tone": <1-5>, "safety": <1-5>}}'''

def judge_reply(customer_text, checklist, reply):
    prompt = JUDGE_RUBRIC.format(customer_text=customer_text, checklist=checklist, reply=reply)
    raw = llm_call(prompt, max_tokens=100)
    try:
        return json.loads(raw)
    except Exception:
        return {"grounding": None, "actionability": None, "tone": None, "safety": None}

# NOTE: run_agent(...) from notebook 03 needed here to actually generate replies for golden_df first.
# golden_df['generated_reply'] = golden_df['customer_text'].apply(lambda t: run_agent(t)['reply'])
# golden_df['judge_scores'] = golden_df.apply(lambda r: judge_reply(r['customer_text'], r['good_reply_checklist'], r['generated_reply']), axis=1)


### B4 — Human vs. judge agreement (mandatory)

In [ ]:
calib = golden_df.sample(n=min(JUDGE_CALIBRATION_SIZE, len(golden_df)), random_state=RANDOM_STATE).copy()
calib.to_csv(os.path.join(GOLDEN_DIR, "judge_calibration_TEMPLATE.csv"), index=False)
print("Hand-score these", len(calib), "rows yourself on the same 1-5 axes, save as judge_calibration_scored.csv")


In [ ]:
# After you've hand-scored judge_calibration_scored.csv (add columns human_grounding, human_actionability, etc.):
scored = pd.read_csv(os.path.join(GOLDEN_DIR, "judge_calibration_scored.csv"))

from sklearn.metrics import cohen_kappa_score
for axis in ["grounding", "actionability", "tone", "safety"]:
    human_col, judge_col = f"human_{axis}", f"judge_{axis}"
    if human_col in scored.columns and judge_col in scored.columns:
        kappa = cohen_kappa_score(scored[human_col], scored[judge_col], weights='linear')
        agreement_pct = (scored[human_col] == scored[judge_col]).mean()
        print(f"{axis}: kappa={kappa:.2f}, exact agreement={agreement_pct:.2f}")


---
## Write-up reminders (goes into REPORT.md, not this notebook)
- Compare all 3 classifier tiers + escalation accuracy + false auto-handle rate in one table.
- Pull the worst-scoring rows across intent / retrieval / generation *separately* for your top-5
  failure modes with real examples and a hypothesis each.
- "What's misleading about my headline number": small self-labeled golden set, LLM-judge possibly
  favoring LLM-style phrasing over authentic brand voice, and check for retrieval/golden-set
  overlap (data leakage) explicitly:
```python
overlap = set(golden_df['customer_text']) & set(pairs_df['customer_text_clean'])
print("Golden/retrieval overlap:", len(overlap))
```
